# OpSet Versions — Hands-On Application

## Objective

Master ONNX opset version management: inspecting environments, building at specific opsets,
querying operator history, converting between versions, and building compatibility tools.

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Setup](#1-setup) | Imports and environment info |
| 2 | [Exercise 1: Environment Inspection](#2-exercise-1) | Current opset, available ops |
| 3 | [Exercise 2: Build at Different OpSets](#3-exercise-2) | Multi-version model construction |
| 4 | [Exercise 3: Query since_version](#4-exercise-3) | Operator introduction versions |
| 5 | [Exercise 4: Version Conversion](#5-exercise-4) | Upgrade and downgrade models |
| 6 | [Exercise 5: Multi-Domain Opset Imports](#6-exercise-5) | Standard + ML + custom |
| 7 | [Exercise 6: Behavior Changes Across Versions](#7-exercise-6) | Softmax axis migration |
| 8 | [Exercise 7: Compatibility Matrix Builder](#8-exercise-7) | Runtime fleet analysis |
| 9 | [Challenge: OpSet Migration Tool](#9-challenge) | Automated upgrade pipeline |
| 10 | [Summary](#10-summary) | Skills review |

In [ ]:
# 1. Setup <a id="1-setup"></a>
# !pip install onnx numpy matplotlib --quiet

import onnx
from onnx import helper, TensorProto, checker, defs, numpy_helper
from onnx import version_converter, shape_inference
from onnx.reference import ReferenceEvaluator
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

print(f"ONNX version: {onnx.__version__}")
print(f"Default opset: {defs.onnx_opset_version()}")
print(f"IR version: {onnx.IR_VERSION}")

## 2. Exercise 1: Environment Inspection <a id="2-exercise-1"></a>

Before working with opset versions, understand what your environment supports.

Key questions:
- What is the maximum opset version supported?
- How many operators exist in each domain?
- Which operators were introduced most recently?

In [ ]:
# Get all schemas with history
all_schemas = defs.get_all_schemas_with_history()

# Organize by domain
domain_ops = defaultdict(set)
domain_schemas = defaultdict(list)
for s in all_schemas:
    d = s.domain or "(default)"
    domain_ops[d].add(s.name)
    domain_schemas[d].append(s)

print("ONNX Environment Summary")
print("═" * 55)
print(f"  Max opset version: {defs.onnx_opset_version()}")
print(f"  IR version:        {onnx.IR_VERSION}")
print(f"  Total schemas:     {len(all_schemas)}")
print(f"\n  {'Domain':<25} | {'Operators':>10} | {'Schemas':>8}")
print(f"  {'-'*50}")
for d in sorted(domain_ops.keys()):
    print(f"  {d:<25} | {len(domain_ops[d]):>10} | {len(domain_schemas[d]):>8}")

# Most recently introduced operators
default_schemas = [s for s in all_schemas if s.domain == "" or s.domain == "ai.onnx"]
latest_introductions = {}
for s in default_schemas:
    if s.name not in latest_introductions or s.since_version > latest_introductions[s.name]:
        latest_introductions[s.name] = s.since_version

max_v = defs.onnx_opset_version()
recent_ops = [(name, v) for name, v in latest_introductions.items() if v >= max_v - 3]
print(f"\nRecently introduced/updated operators (opset >= {max_v-3}):")
for name, v in sorted(recent_ops, key=lambda x: -x[1])[:15]:
    print(f"  {name:<25} since opset {v}")

## 3. Exercise 2: Build at Different OpSets <a id="3-exercise-2"></a>

Build the same model architecture at multiple opset versions and verify validity.

The basic operators (MatMul, Add, Relu) exist since early opsets, so simple models
work across a wide range. Complex models with newer ops have a minimum opset floor:

$$v_{\min} = \max_{\text{op} \in G}\; \text{since\_version}(\text{op})$$

In [ ]:
def build_at_opset(opset: int) -> tuple:
    """Build a simple MLP at the specified opset. Returns (model, status)."""
    np.random.seed(42)
    W = numpy_helper.from_array(np.random.randn(10, 5).astype(np.float32) * 0.1, "W")
    b = numpy_helper.from_array(np.random.randn(5).astype(np.float32) * 0.01, "b")

    nodes = [
        helper.make_node("MatMul", ["X", "W"], ["mm"]),
        helper.make_node("Add", ["mm", "b"], ["z"]),
        helper.make_node("Relu", ["z"], ["Y"]),
    ]

    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 10])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [2, 5])
    graph = helper.make_graph(nodes, f"mlp_v{opset}", [X], [Y], initializer=[W, b])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])

    try:
        checker.check_model(model)
        return model, "valid"
    except Exception as e:
        return model, f"invalid: {str(e)[:60]}"


# Test across all opsets
test_opsets = list(range(7, defs.onnx_opset_version() + 2))
results = []

print(f"{'OpSet':>6} | {'Status':<10} | {'Inference':>9} | Notes")
print("-" * 55)

x_test = np.random.randn(2, 10).astype(np.float32)
for v in test_opsets:
    model, status = build_at_opset(v)
    infer_ok = False

    if status == "valid":
        try:
            ev = ReferenceEvaluator(model)
            y = ev.run(None, {"X": x_test})[0]
            infer_ok = True
        except Exception:
            pass

    results.append({"opset": v, "valid": status == "valid", "infer": infer_ok})
    s_sym = "✓" if status == "valid" else "✗"
    i_sym = "✓" if infer_ok else "✗"
    notes = "" if status == "valid" else status[9:]
    print(f"{v:>6} | {s_sym:<10} | {i_sym:>9} | {notes}")

valid_count = sum(1 for r in results if r["infer"])
print(f"\n{valid_count}/{len(test_opsets)} opset versions produce valid inference.")

# Visualize
fig, ax = plt.subplots(figsize=(12, 3))
colors = ["#4CAF50" if r["infer"] else ("#FFA500" if r["valid"] else "#F44336") for r in results]
ax.bar([r["opset"] for r in results], [1] * len(results), color=colors, edgecolor="black")
ax.set_xticks([r["opset"] for r in results])
ax.set_yticks([])
ax.set_xlabel("OpSet Version")
ax.set_title("Model Validity Across OpSet Versions", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Exercise 3: Query since_version <a id="4-exercise-3"></a>

For any operator, `since_version` tells you the minimum opset where it's available.
This is critical for determining the minimum opset for a model.

The minimum required opset for a graph $G$ is:
$$v_{\text{min}}(G) = \max_{\text{op} \in G}\; \text{since\_version}(\text{op})$$

In [ ]:
def query_since_version(op_name: str, domain: str = "") -> dict:
    """Get all version history for an operator."""
    history = []
    for s in all_schemas:
        if s.name == op_name and (s.domain == domain or (s.domain == "" and domain == "")):
            history.append(s.since_version)
    history = sorted(set(history))

    current = defs.onnx_opset_version()
    try:
        schema = defs.get_schema(op_name, current, domain)
        current_since = schema.since_version
    except Exception:
        current_since = None

    return {
        "op": op_name,
        "domain": domain or "(default)",
        "introduced": history[0] if history else None,
        "revisions": history,
        "n_revisions": len(history),
        "current_since": current_since,
    }


# Query common operators
operators_to_check = [
    "MatMul", "Add", "Relu", "Conv", "BatchNormalization",
    "Softmax", "LayerNormalization", "Reshape", "Squeeze",
    "Gather", "Resize", "Pad", "ReduceMean", "ScatterND",
    "GroupNormalization",
]

print(f"{'Operator':<25} | {'Introduced':>10} | {'Current':>7} | {'Revisions':>9} | History")
print("-" * 85)

op_data = []
for op in operators_to_check:
    info = query_since_version(op)
    intro = info["introduced"] or "N/A"
    curr = info["current_since"] or "N/A"
    revs = info["revisions"]
    print(f"{op:<25} | {str(intro):>10} | {str(curr):>7} | {info['n_revisions']:>9} | {revs}")
    op_data.append(info)

# Find the most-revised operators
print(f"\nMost-revised operators indicate semantic complexity and breaking changes.")

# Compute minimum opset for a model
def compute_min_opset(model: onnx.ModelProto) -> tuple:
    """Compute minimum opset and identify bottleneck operators."""
    max_opset = defs.onnx_opset_version()
    declared = model.opset_import[0].version

    op_versions = {}
    for node in model.graph.node:
        try:
            schema = defs.get_schema(node.op_type, declared, node.domain or "")
            op_versions[node.op_type] = schema.since_version
        except Exception:
            op_versions[node.op_type] = None

    valid = {k: v for k, v in op_versions.items() if v is not None}
    min_required = max(valid.values()) if valid else 1
    bottleneck = [op for op, v in valid.items() if v == min_required]
    headroom = declared - min_required

    return min_required, bottleneck, headroom, op_versions


# Test on our models
model, _ = build_at_opset(17)
min_req, bottleneck, headroom, versions = compute_min_opset(model)
print(f"\nModel declared at opset 17:")
print(f"  Min required: {min_req}")
print(f"  Bottleneck:   {bottleneck}")
print(f"  Headroom:     {headroom} versions")
print(f"  Could safely declare opset {min_req} for wider compatibility!")

## 5. Exercise 4: Version Conversion <a id="5-exercise-4"></a>

Use `onnx.version_converter.convert_version()` to upgrade and downgrade models.

**Safety rules:**
- Upgrades are generally safe (new opsets are backward-compatible)
- Downgrades may fail (target opset might not support all ops)

After conversion, verify: $\forall \mathbf{x}: \|f_{v_s}(\mathbf{x}) - f_{v_t}(\mathbf{x})\|_\infty < \epsilon$

In [ ]:
def convert_and_verify(model: onnx.ModelProto, target: int,
                       test_inputs: dict) -> dict:
    """Convert model to target opset and verify equivalence."""
    source = model.opset_import[0].version
    result = {"source": source, "target": target}

    try:
        converted = version_converter.convert_version(model, target)
        checker.check_model(converted)
        result["status"] = "ok"
        result["nodes_src"] = len(model.graph.node)
        result["nodes_dst"] = len(converted.graph.node)

        # Semantic verification
        try:
            y_src = ReferenceEvaluator(model).run(None, test_inputs)[0]
            y_dst = ReferenceEvaluator(converted).run(None, test_inputs)[0]
            result["max_diff"] = float(np.abs(y_src - y_dst).max())
            result["equivalent"] = result["max_diff"] < 1e-5
        except Exception as e:
            result["max_diff"] = None
            result["equivalent"] = "eval_error"

        return converted, result
    except Exception as e:
        result["status"] = "failed"
        result["error"] = str(e)[:80]
        return None, result


# Build source model at opset 13
np.random.seed(42)
W1 = numpy_helper.from_array(np.random.randn(10, 20).astype(np.float32) * 0.1, "W1")
b1 = numpy_helper.from_array(np.random.randn(20).astype(np.float32) * 0.01, "b1")
W2 = numpy_helper.from_array(np.random.randn(20, 5).astype(np.float32) * 0.1, "W2")
b2 = numpy_helper.from_array(np.random.randn(5).astype(np.float32) * 0.01, "b2")

nodes = [
    helper.make_node("MatMul", ["X", "W1"], ["mm1"]),
    helper.make_node("Add", ["mm1", "b1"], ["h1"]),
    helper.make_node("Relu", ["h1"], ["r1"]),
    helper.make_node("MatMul", ["r1", "W2"], ["mm2"]),
    helper.make_node("Add", ["mm2", "b2"], ["Y"]),
]
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 5])
graph = helper.make_graph(nodes, "mlp", [X], [Y], initializer=[W1, b1, W2, b2])
source_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
checker.check_model(source_model)

x_test = {"X": np.random.randn(4, 10).astype(np.float32)}
targets = [9, 11, 13, 15, 17, 18, 19]

print(f"Source: opset 13, {len(source_model.graph.node)} nodes")
print(f"\n{'Target':>7} | {'Status':<8} | {'Nodes':>10} | {'Max Diff':>10} | {'Equiv':>6}")
print("-" * 55)

conversion_log = []
for target in targets:
    _, result = convert_and_verify(source_model, target, x_test)
    conversion_log.append(result)

    status = result["status"]
    if status == "ok":
        nodes_str = f"{result['nodes_src']}→{result['nodes_dst']}"
        diff_str = f"{result['max_diff']:.2e}" if result["max_diff"] is not None else "N/A"
        eq_str = "✓" if result.get("equivalent") is True else str(result.get("equivalent", "?"))
    else:
        nodes_str = "—"
        diff_str = "—"
        eq_str = "✗"
    print(f"{target:>7} | {status:<8} | {nodes_str:>10} | {diff_str:>10} | {eq_str:>6}")

successful = sum(1 for r in conversion_log if r["status"] == "ok")
print(f"\n{successful}/{len(targets)} conversions successful.")

## 6. Exercise 5: Multi-Domain Opset Imports <a id="6-exercise-5"></a>

ONNX models can import operators from multiple domains, each versioned independently:

$$\text{opset\_imports}(M) = \{(d_1, v_1), (d_2, v_2), \ldots, (d_n, v_n)\}$$

Common domains:
- `""` — default ONNX operators (Conv, MatMul, etc.)
- `"ai.onnx.ml"` — ML operators (tree ensembles, SVMs)
- Custom domains — user-defined operators

In [ ]:
# Build a model with multiple domain imports
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 5])
W = numpy_helper.from_array(np.random.randn(10, 5).astype(np.float32), "W")

nodes = [
    helper.make_node("MatMul", ["X", "W"], ["mm"]),
    helper.make_node("Relu", ["mm"], ["Y"]),
]

graph = helper.make_graph(nodes, "multi_domain", [X], [Y], initializer=[W])
model = helper.make_model(
    graph,
    opset_imports=[
        helper.make_opsetid("", 17),
        helper.make_opsetid("ai.onnx.ml", 3),
        helper.make_opsetid("com.mycompany.ops", 1),
    ],
)

print("Multi-domain model:")
print(f"  {'Domain':<25} | {'Version':>7}")
print(f"  {'-'*38}")
for oi in model.opset_import:
    d = oi.domain or "(default/ai.onnx)"
    print(f"  {d:<25} | {oi.version:>7}")

# Manipulate opset imports
def get_opset_version(model, domain=""):
    for oi in model.opset_import:
        if oi.domain == domain:
            return oi.version
    return None

def set_opset_version(model, domain, version):
    for oi in model.opset_import:
        if oi.domain == domain:
            oi.version = version
            return
    new_oi = model.opset_import.add()
    new_oi.domain = domain
    new_oi.version = version

# Verify access
assert get_opset_version(model, "") == 17
assert get_opset_version(model, "ai.onnx.ml") == 3
assert get_opset_version(model, "com.mycompany.ops") == 1

# Update a domain version
set_opset_version(model, "ai.onnx.ml", 4)
assert get_opset_version(model, "ai.onnx.ml") == 4
print(f"\nUpdated ai.onnx.ml to v4. ✓")
print(f"Domains: {[(oi.domain or 'default', oi.version) for oi in model.opset_import]}")

## 7. Exercise 6: Behavior Changes Across Versions <a id="7-exercise-6"></a>

Some operators change **semantics** across opset versions. The most notorious example:

**Softmax at opset < 13:**
$$\text{softmax}(x, \text{axis}=1) \text{ with implicit flattening beyond axis}$$

**Softmax at opset ≥ 13:**
$$\text{softmax}(x, \text{axis}=-1) \text{ standard per-axis normalization}$$

This is a **silent correctness bug** if not handled properly.

In [ ]:
def test_softmax_behavior(opset: int, input_shape: list, axis: int = None):
    """Test Softmax behavior at specific opset."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, input_shape)
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, input_shape)

    kwargs = {}
    if axis is not None:
        kwargs["axis"] = axis

    node = helper.make_node("Softmax", ["X"], ["Y"], **kwargs)
    graph = helper.make_graph([node], "sm", [X], [Y])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", opset)])

    np.random.seed(42)
    x = np.random.randn(*input_shape).astype(np.float32)

    try:
        y = ReferenceEvaluator(model).run(None, {"X": x})[0]
        return y, None
    except Exception as e:
        return None, str(e)[:60]


# Compare 2D behavior
print("Softmax Behavior Analysis (no explicit axis)")
print("═" * 60)

shape_2d = [2, 5]
shape_4d = [1, 3, 4, 4]

for shape, label in [(shape_2d, "2D [2,5]"), (shape_4d, "4D [1,3,4,4]")]:
    print(f"\n  Input shape: {label}")
    results = {}
    for opset in [11, 12, 13, 17]:
        y, err = test_softmax_behavior(opset, shape)
        if y is not None:
            results[opset] = y
            sum_last = y.sum(axis=-1)
            print(f"    opset {opset:>2}: sum(axis=-1) range = [{sum_last.min():.4f}, {sum_last.max():.4f}]")
        else:
            print(f"    opset {opset:>2}: error - {err}")

    # Detect if behavior changed
    if len(results) >= 2:
        opsets = sorted(results.keys())
        for i in range(len(opsets) - 1):
            diff = np.abs(results[opsets[i]] - results[opsets[i+1]]).max()
            if diff > 1e-6:
                print(f"    ⚠ BEHAVIOR CHANGE between opset {opsets[i]} and {opsets[i+1]}! (diff={diff:.4f})")

# Safe practice: always specify axis explicitly
print("\n\nBest Practice: Always specify axis explicitly!")
y_safe, _ = test_softmax_behavior(17, [2, 5], axis=-1)
assert y_safe is not None
assert np.allclose(y_safe.sum(axis=-1), 1.0)
print(f"  Softmax(axis=-1): sum per row = {y_safe.sum(axis=-1)} ✓")

## 8. Exercise 7: Compatibility Matrix Builder <a id="8-exercise-7"></a>

Build a tool that generates a compatibility matrix showing which runtime environments
can execute a model.

Compatibility rule:
$$\text{Compatible}(M, R) \iff \forall d \in \text{domains}(M): v_M(d) \leq v_R(d)$$

In [ ]:
RUNTIME_PROFILES = {
    "ORT 1.10 (Edge)": {"": 15},
    "ORT 1.12": {"": 17, "ai.onnx.ml": 3},
    "ORT 1.14": {"": 18, "ai.onnx.ml": 3},
    "ORT 1.16": {"": 19, "ai.onnx.ml": 3},
    "ORT 1.18": {"": 20, "ai.onnx.ml": 4},
    "TensorRT 8.6": {"": 17},
    "OpenVINO 2023": {"": 17, "ai.onnx.ml": 2},
    "Core ML": {"": 15},
}


def build_compatibility_matrix(opset_range: list) -> np.ndarray:
    """Build a compatibility matrix: models (rows) x runtimes (cols)."""
    runtimes = list(RUNTIME_PROFILES.keys())
    matrix = np.zeros((len(opset_range), len(runtimes)), dtype=int)

    for i, opset in enumerate(opset_range):
        for j, (name, profile) in enumerate(RUNTIME_PROFILES.items()):
            # Check if model at this opset is compatible
            compatible = opset <= profile.get("", 0)
            matrix[i, j] = 1 if compatible else 0

    return matrix, opset_range, runtimes


opset_range = list(range(11, 21))
matrix, opsets, runtimes = build_compatibility_matrix(opset_range)

# Print matrix
print(f"{'OpSet':>6}", end="")
for r in runtimes:
    print(f" | {r[:8]:>8}", end="")
print(" | Coverage")
print("-" * (6 + 11 * len(runtimes) + 10))

for i, opset in enumerate(opsets):
    print(f"{opset:>6}", end="")
    for j in range(len(runtimes)):
        sym = "✓" if matrix[i, j] else "✗"
        print(f" | {sym:>8}", end="")
    coverage = matrix[i].sum()
    print(f" | {coverage}/{len(runtimes)}")

# Optimal opset (maximum coverage)
coverages = matrix.sum(axis=1)
best_idx = np.argmax(coverages)
print(f"\nOptimal opset for max coverage: {opsets[best_idx]} ({coverages[best_idx]}/{len(runtimes)} runtimes)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(matrix, cmap="RdYlGn", aspect="auto")
axes[0].set_xticks(range(len(runtimes)))
axes[0].set_xticklabels([r[:10] for r in runtimes], rotation=45, ha="right", fontsize=8)
axes[0].set_yticks(range(len(opsets)))
axes[0].set_yticklabels([f"opset {v}" for v in opsets], fontsize=9)
axes[0].set_title("Compatibility Matrix", fontweight="bold")
for i in range(len(opsets)):
    for j in range(len(runtimes)):
        axes[0].text(j, i, "✓" if matrix[i, j] else "", ha="center", va="center", fontsize=8)

# Coverage plot
colors = ["#4CAF50" if c == max(coverages) else "#2196F3" for c in coverages]
axes[1].bar(range(len(opsets)), coverages, color=colors)
axes[1].set_xticks(range(len(opsets)))
axes[1].set_xticklabels([str(v) for v in opsets])
axes[1].set_xlabel("Model OpSet")
axes[1].set_ylabel("Compatible Runtimes")
axes[1].set_title("Fleet Coverage by OpSet", fontweight="bold")
axes[1].axhline(len(runtimes), color="gray", linestyle="--", alpha=0.5, label="Total runtimes")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Challenge: OpSet Migration Tool <a id="9-challenge"></a>

Build a comprehensive migration tool that:
1. Analyzes a model's minimum required opset
2. Attempts conversion to a target opset
3. Verifies numerical equivalence
4. Reports compatibility changes
5. Generates a migration report

In [ ]:
class OpSetMigrationTool:
    """Tool for safely migrating ONNX models between opset versions."""

    def __init__(self, runtime_profiles: dict = None):
        self.profiles = runtime_profiles or RUNTIME_PROFILES

    def analyze(self, model: onnx.ModelProto) -> dict:
        """Analyze current model state."""
        declared = model.opset_import[0].version
        min_req, bottleneck, headroom, op_versions = compute_min_opset(model)

        # Current compatibility
        compatible_runtimes = [
            name for name, profile in self.profiles.items()
            if declared <= profile.get("", 0)
        ]

        return {
            "declared_opset": declared,
            "min_required": min_req,
            "headroom": headroom,
            "bottleneck_ops": bottleneck,
            "op_versions": op_versions,
            "compatible_runtimes": compatible_runtimes,
            "coverage": len(compatible_runtimes),
        }

    def migrate(self, model: onnx.ModelProto, target_opset: int,
                test_input: dict = None) -> dict:
        """Attempt migration to target opset."""
        analysis_before = self.analyze(model)

        report = {
            "source_opset": analysis_before["declared_opset"],
            "target_opset": target_opset,
            "before": analysis_before,
        }

        # Attempt conversion
        try:
            converted = version_converter.convert_version(model, target_opset)
            checker.check_model(converted)
            report["conversion"] = "success"
            report["model"] = converted

            # Analyze after
            analysis_after = self.analyze(converted)
            report["after"] = analysis_after
            report["coverage_change"] = analysis_after["coverage"] - analysis_before["coverage"]

            # Verify equivalence
            if test_input:
                try:
                    y_before = ReferenceEvaluator(model).run(None, test_input)[0]
                    y_after = ReferenceEvaluator(converted).run(None, test_input)[0]
                    report["max_diff"] = float(np.abs(y_before - y_after).max())
                    report["equivalent"] = report["max_diff"] < 1e-5
                except Exception as e:
                    report["equivalent"] = f"eval error: {str(e)[:40]}"

        except Exception as e:
            report["conversion"] = "failed"
            report["error"] = str(e)[:100]

        return report

    def print_report(self, report: dict):
        """Print a formatted migration report."""
        print("\n" + "═" * 60)
        print(f"  MIGRATION REPORT: opset {report['source_opset']} → {report['target_opset']}")
        print("═" * 60)
        print(f"  Conversion: {report['conversion']}")

        if report["conversion"] == "success":
            before = report["before"]
            after = report["after"]
            print(f"  Coverage:   {before['coverage']} → {after['coverage']} runtimes "
                  f"({report['coverage_change']:+d})")
            if "equivalent" in report:
                eq = "✓" if report["equivalent"] is True else str(report["equivalent"])
                print(f"  Equivalent: {eq}")
                if isinstance(report.get("max_diff"), float):
                    print(f"  Max diff:   {report['max_diff']:.2e}")
            print(f"\n  New compatible runtimes:")
            new_runtimes = set(after["compatible_runtimes"]) - set(before["compatible_runtimes"])
            lost_runtimes = set(before["compatible_runtimes"]) - set(after["compatible_runtimes"])
            for r in new_runtimes:
                print(f"    + {r}")
            for r in lost_runtimes:
                print(f"    - {r}")
            if not new_runtimes and not lost_runtimes:
                print(f"    (no change)")
        else:
            print(f"  Error: {report.get('error', 'unknown')}")


# Demo: migrate a model
tool = OpSetMigrationTool()

source = source_model  # opset 13 model from earlier
test_in = {"X": np.random.randn(4, 10).astype(np.float32)}

# Try migrating to different targets
for target in [11, 15, 17, 19]:
    report = tool.migrate(source, target, test_in)
    tool.print_report(report)

## 10. Summary <a id="10-summary"></a>

| Exercise | Skill | Key Formula |
|----------|-------|---------|
| 1. Environment | Inspect available ops and versions | `defs.onnx_opset_version()` |
| 2. Multi-OpSet Build | Construct models at any opset | Validity across versions |
| 3. since_version | Query operator introduction history | $v_{\min} = \max \text{since\_version}$ |
| 4. Conversion | Safe upgrade/downgrade | $G_{v_s} \to G_{v_t}$ |
| 5. Multi-Domain | Independent domain versioning | $\{(d_i, v_i)\}$ |
| 6. Behavior Changes | Detect semantic differences | Softmax axis migration |
| 7. Compatibility | Runtime fleet analysis | $\forall d: v_M(d) \leq v_R(d)$ |
| Challenge | Migration Tool | End-to-end pipeline |

### Key Takeaways

1. **Export at the lowest viable opset** for maximum deployment coverage
2. **Always verify equivalence** after version conversion: $\|f_{v_s} - f_{v_t}\| < \epsilon$
3. **Watch for semantic changes** (Softmax axis, Squeeze axes→input) — these are silent bugs
4. **Build compatibility checking** into your CI/CD pipeline